# Работа с числами и строками на примере алгоритма Луна
---
М.А. Гейне (mike.geine@gmail.com)

Цель — пройти от строки к проверяемому результату: определить допустимые данные,
разложить алгоритм на преобразования и сравнить цикл с функциональным стилем.
Python 3.14; примеры не требуют сети или сторонних библиотек.

Основной маршрут: строки и байты → пошаговый Luhn → договор о данных →
две реализации → известные ответы и ошибки формата. Ячейки отмечены
комментарием `Основной маршрут` и тегом `core` и выполняются независимо от справочника.
Нормализация Unicode, регулярные выражения и дополнительные свойства — расширение.


## Строки, Unicode и байты

Строка Python — последовательность кодовых точек Unicode. Один видимый символ
может состоять из нескольких кодовых точек. При передаче по сети или записи в
файл строку кодируют в байты; для обратного преобразования нужна кодировка.

In [ ]:
# Основной маршрут
s = "Привет, мир!"
print(f"Тип: {type(s)}, Длина: {len(s)}")

# Кодируем строку в байты, используя кодировку UTF-8
b = s.encode('utf-8')
print(f"Тип: {type(b)}, Длина: {len(b)}")
print(f"Байтовое представление: {b}")

# Декодируем байты обратно в строку
s_decoded = b.decode('utf-8')
print(f"Раскодированная строка: {s_decoded}")

# Попытка декодировать с неверной кодировкой вызовет ошибку
try:
    b.decode('ascii')
except UnicodeDecodeError as e:
    print(f"\nОшибка: {e}")

### Обработка ошибок кодирования

При работе с реальными данными часто встречаются проблемы с кодировками. Параметр `errors` помогает их обработать.

In [ ]:
# Эта байтовая строка содержит и ASCII, и не-ASCII символы (кириллица в UTF-8)
b_mixed = b'hello \xd0\xbf\xd1\x80\x69\x76\x65\x74 world'

# errors='strict' (по умолчанию) - вызовет ошибку при декодировании в ascii
# errors='replace' - заменит некорректные для ascii символы на 'U+FFFD'
print(f"'replace': {b_mixed.decode('ascii', errors='replace')}")

# errors='ignore' - просто проигнорирует (выбросит) некорректные символы
print(f"'ignore': {b_mixed.decode('ascii', errors='ignore')}")

### Нормализация Unicode · дополнительно

Внешне одинаковый текст может иметь разные представления. Нормализация помогает
сравнивать их. Удаление диакритики — отдельное преобразование с потерей информации,
которое допустимо далеко не в каждой задаче.

In [ ]:
import unicodedata

text = "Crème brûlée"

def remove_accents(input_str):
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return u"".join([c for c in nfkd_form if not unicodedata.combining(c)])

cleaned_text = remove_accents(text)
print(f"Оригинал: {text}")
print(f"Без диакритики: {cleaned_text}")

# Также можно получить информацию о символе
print(f"Имя символа 'é': {unicodedata.name('é')}")

In [ ]:
composed = "é"
decomposed = "e\u0301"
composed == decomposed, len(composed), len(decomposed), unicodedata.normalize("NFC", decomposed) == composed

## Регулярные выражения · справочник

Шаблон описывает форму текста. Совпадение с шаблоном ещё не доказывает смысловую
корректность данных: например, запись IP-адреса может содержать число больше 255.
Для проверки строки целиком используйте `re.fullmatch`.

Расшифровка:

- `U` - поиск символа U (или любого другого символа, указанного буквально)
- `a-z` - любой символ в интервале
- `[abc]` - любой из символов a, b или c
- `[^abc]` - любой из символов, кроме abc
- `.` - любой символ
- `\d` - любая цифра (0-9)
- `\D` - любой символ, кроме цифры
- `\w` - любой символ слова ([a-zA-Z0-9_])
- `\s` - любой символ пробела
- `{4}` - количество повторений предыдущего токена (4 раза)
- `+` - предыдущий токен должен повториться 1 или более раз
- `*` - предыдущий токен должен повториться 0 или более раз
- `()` - группа захвата

См. также: [https://regex101.com/](https://regex101.com/)


### Именованные группы для извлечения данных

Использование `(?P<name>...)` делает код более читаемым и надежным, так как вы обращаетесь к результатам по имени, а не по индексу.

In [ ]:
import re
log_line = '2023-10-27 10:30:00 - ERROR - User [johndoe] attempted login from IP 192.168.1.100'

# Используем именованные группы для каждого извлекаемого элемента
pattern = r'(?P<date>\d{4}-\d{2}-\d{2})\s+(?P<time>\d{2}:\d{2}:\d{2})\s+-\s+(?P<level>\w+)\s+-\s+User\s+\[(?P<username>\w+)\].*IP\s+(?P<ip>\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})'

match = re.search(pattern, log_line)
if match:
    log_data = match.groupdict()
    print(log_data)
    print(f"Пользователь {log_data['username']} совершил действие с уровнем {log_data['level']}")

### "Болтливые" (verbose) регулярные выражения

Для сложных шаблонов можно использовать флаг `re.VERBOSE` (или `re.X`), чтобы добавлять пробелы и комментарии, делая выражение понятным.

In [ ]:
# Учебный шаблон распространённой формы адреса; не полная проверка email
email_pattern_verbose = re.compile(r"""
    [a-zA-Z0-9._%+-]+  # локальная часть
    @
    [a-zA-Z0-9.-]+     # домен
    \.
    [a-zA-Z]{2,}      # доменная зона
""", re.VERBOSE)
email = "test.user+alias@example.com"
bool(email_pattern_verbose.fullmatch(email))

### Просмотр вперед/назад (Lookarounds)

Lookarounds позволяют проверять наличие символов до или после совпадения, *не включая* их в само совпадение. Это полезно для извлечения данных в определенном контексте.

- `(?=...)` - Positive Lookahead (просмотр вперед): ищет совпадение, только если *за ним* следует `...`.
- `(?<=...)` - Positive Lookbehind (просмотр назад): ищет совпадение, только если *перед ним* есть `...`.

In [ ]:
text = "Товар А стоит $10.99, товар Б — $15.50. Скидка 5%. Тест $ ого 16.60"
prices = re.findall(r"(?<=\$)[0-9]+\.[0-9]{2}\b", text)
discounts = re.findall(r"[0-9]+(?=%)", text)
assert prices == ["10.99", "15.50"]
prices, discounts

### Ошибка: жадный квантификатор захватывает лишнее

`.*` допускает любые символы и старается захватить как можно больше.
Поэтому следующий шаблон возвращает большой фрагмент, а не отдельные цены.
Исправление выше описывает именно цифры, точку и два знака дробной части.

In [ ]:
wrong_prices = re.findall(r"(?<=\$).*\d+\.\d+", text)
assert wrong_prices != ["10.99", "15.50"]
wrong_prices, prices

## Алгоритм Луна

Luhn проверяет контрольную сумму цифрового идентификатора. Он помогает обнаруживать
ошибки ввода, но не подтверждает существование номера или принадлежность владельцу.
Все номера ниже — учебные примеры.

1. Идём справа налево. Крайнюю правую цифру (контрольную) оставляем без изменения.
2. Удваиваем каждую вторую цифру. Если результат больше 9, вычитаем 9
   (это равносильно сумме двух цифр результата).
3. Складываем полученные значения.
4. Проверка успешна, если сумма кратна 10.

Номер храним строкой: ведущие нули — часть идентификатора, арифметика над самим
номером нам не нужна.

In [ ]:
# Основной маршрут
card_number = '4111-1111-4555-1142'

0. Подготовка номера карты

In [ ]:
# Основной маршрут
card_translation = str.maketrans({'-': '', ' ': ''})
translated_card_number = card_number.translate(card_translation)
translated_card_number

### 1. Цифры на нечётных позициях справа, начиная с 1

In [ ]:
# Основной маршрут
card_number_reversed = translated_card_number[::-1]
card_number_reversed

In [ ]:
# Основной маршрут
odd_digits = card_number_reversed[::2]
odd_digits

In [ ]:
# Основной маршрут
sum_of_odd_digits = 0

for digit in odd_digits:
  sum_of_odd_digits += int(digit)
sum_of_odd_digits

### 2. Цифры на чётных позициях справа

In [ ]:
# Основной маршрут
even_digits = card_number_reversed[1::2]
even_digits

In [ ]:
# Основной маршрут
sum_of_even_digits = 0

for digit in even_digits:
  number = int(digit) * 2
  if number >= 10:
    number = (number // 10) + (number % 10)
  sum_of_even_digits += number

sum_of_even_digits



3. Подсчёт суммы

In [ ]:
# Основной маршрут
total = sum_of_even_digits + sum_of_odd_digits
total

In [ ]:
# Основной маршрут
total % 10 == 0

Чётность относится к **позиции**, а не к значению цифры. Если отсчитывать позиции
слева, правило зависит от длины номера. Обход справа устраняет эту зависимость.

### Ошибка: пустой ввод случайно проходит проверку

Сумма пустой последовательности равна нулю. Поэтому одной проверки остатка
недостаточно: пустая строка ошибочно выглядит корректной. Далее проверим
формат входа отдельно от контрольной суммы.

In [ ]:
empty_digits = []
sum(empty_digits), sum(empty_digits) % 10 == 0

## Оформим в функции: сначала договор о данных

`clear_card_number` удаляет только обычные пробелы и дефисы. После этого допустима
непустая строка из ASCII-цифр `0`–`9`. Остальные символы и пустой ввод вызывают
`ValueError`: это ошибка формата. Для строки допустимого формата `verify_luhn`
возвращает `True` или `False`: это результат проверки контрольной суммы.

Длины конкретных банковских номеров здесь не проверяем: реализуем общий алгоритм.
Аннотация `str` описывает договор функции, но сама не проверяет тип во время исполнения.

In [ ]:
# Основной маршрут
# «Цифра» в Unicode шире договора нашего идентификатора
[(text, text.isdigit(), text.isdecimal(), text.isascii())
 for text in ["123", "١٢٣", "１２３", "²", ""]]

In [ ]:
# Основной маршрут
def clear_card_number(card_number: str) -> str:
    return card_number.translate(str.maketrans({"-": "", " ": ""}))


def require_digits(text: str) -> None:
    if not text or not all("0" <= char <= "9" for char in text):
        raise ValueError("Ожидалась непустая строка ASCII-цифр")


def verify_luhn(str_of_digits: str) -> bool:
    require_digits(str_of_digits)
    reversed_number = str_of_digits[::-1]
    total = 0
    for digit in reversed_number[::2]:
        total += int(digit)
    for digit in reversed_number[1::2]:
        doubled = int(digit) * 2
        if doubled > 9:
            doubled -= 9
        total += doubled
    return total % 10 == 0

In [ ]:
# Основной маршрут
card_number = '4111-1111-4555-1142'
clean = clear_card_number(card_number)
verify_luhn(clean)


In [ ]:
# Основной маршрут
card_number = '4111-1111-4555-1143'
clean = clear_card_number(card_number)
verify_luhn(clean)

## Те же преобразования без изменяемого накопителя

**Цифры → позиции справа → вклады цифр → сумма → остаток.**
`sum` уже выражает нужную свёртку. Отдельная функция описывает вклад удвоенной
цифры; генераторное выражение соединяет этапы. Вариант с циклом тоже остаётся
ясным решением: функциональный стиль не требует заменить каждый цикл `reduce`.
Обе функции не меняют вход и не выполняют ввод-вывод.

In [ ]:
# Основной маршрут
def double_digit(digit: int) -> int:
    doubled = digit * 2
    return doubled - 9 if doubled > 9 else doubled


def verify_luhn2(str_of_digits: str) -> bool:
    require_digits(str_of_digits)
    digits_from_right = map(int, reversed(str_of_digits))
    contributions = (
        double_digit(digit) if position % 2 else digit
        for position, digit in enumerate(digits_from_right)
    )
    return sum(contributions) % 10 == 0

In [ ]:
# Основной маршрут
card_number = '4111-1111-4555-1142'
clean = clear_card_number(card_number)
verify_luhn2(clean)

In [ ]:
# Основной маршрут
card_number = '4111-1111-4555-1143'
clean = clear_card_number(card_number)
verify_luhn2(clean)

## Проверки договора и крайних случаев

Совпадения двух реализаций недостаточно: они могут содержать одну ошибку.
Сначала используем примеры с известным ответом, затем проверяем свойства.
`assert` — исполняемое утверждение для этих демонстраций; проверку пользовательского
ввода выше выполняет обычный `if` с `ValueError`.

In [ ]:
# Основной маршрут
known_cases = {
    "79927398713": True,
    "79927398714": False,
    "4111111145551142": True,
    "4111111145551143": False,
    "059": True,  # ведущий ноль сохраняется
    "0": True,   # проходит общую контрольную сумму, не является номером карты
    "1": False,
    "18": True,
}
for digits, expected in known_cases.items():
    assert verify_luhn(digits) is expected
    assert verify_luhn2(digits) is expected
known_cases

In [ ]:
# Основной маршрут
# Ошибку формата не превращаем в «неверную контрольную сумму»
for raw in ["", " -- ", "12x3", "１２３", "١٢٣", "²", "12\t3"]:
    for check in (verify_luhn, verify_luhn2):
        try:
            check(clear_card_number(raw))
        except ValueError:
            pass
        else:
            raise AssertionError(f"Недопустимый ввод принят: {raw!r}")

In [ ]:
# Основной маршрут
raw_number = "7992-7398 713"
clean_number = clear_card_number(raw_number)
assert clear_card_number(clean_number) == clean_number  # идемпотентность очистки
assert raw_number == "7992-7398 713"                     # вход не изменён
assert verify_luhn(clean_number)
assert verify_luhn("059") == verify_luhn2("059")

### Свойство и граница алгоритма

Для фиксированного префикса существует ровно одна контрольная цифра.
Luhn обнаруживает любую замену одной цифры, но не все перестановки:
например, соседние `09` и `90` могут дать одинаковую сумму.

In [ ]:
for prefix in ("0", "05", "123456", "7992739871"):
    valid = [prefix + str(digit) for digit in range(10)
             if verify_luhn2(prefix + str(digit))]
    assert len(valid) == 1
    print(prefix, "→", valid[0])

In [ ]:
number = "79927398713"
for position, old_digit in enumerate(number):
    for new_digit in "0123456789":
        if new_digit != old_digit:
            changed = number[:position] + new_digit + number[position + 1:]
            assert not verify_luhn2(changed)

assert verify_luhn2("091") and verify_luhn2("901")

### Изменение требования: разделители разрешены только на границе

Алгоритм принимает цифры; преобразование пользовательской строки находится снаружи.
Если завтра интерфейс разрешит ещё один разделитель, меняется нормализация,
а проверка суммы остаётся прежней. Нельзя просто выкинуть всё, что не похоже
на цифру: опечатка `12x3` тогда незаметно превратится в другой идентификатор.

In [ ]:
# Основной маршрут
def check_formatted_number(raw: str) -> bool:
    return verify_luhn2(clear_card_number(raw))

assert check_formatted_number("7992-7398 713")
assert not check_formatted_number("7992-7398 714")

Далее функции можно передавать как значения, комбинировать и настраивать их
поведение. Здесь уже есть главное: явный вход, небольшие преобразования,
отделённая нормализация и проверяемый результат.

Документация: [строки](https://docs.python.org/3.14/library/stdtypes.html#text-sequence-type-str),
[итераторы и встроенные функции](https://docs.python.org/3.14/library/functions.html),
[регулярные выражения](https://docs.python.org/3.14/library/re.html).